# 07 - Full Dataset Pipeline (Week 1)

**Goal:** Parse all 50 CholecT50 video JSONs, build graphs for every video, generate dataset-wide statistics, create temporal graph sequences for downstream modeling, and establish train/val/test splits.

In [13]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.triplet_parser import (
    parse_video, filter_valid_triplets, video_summary,
    save_parsed_csv, load_categories, multi_label_analysis,
)
from src.graph_builder import build_graph, save_graph, graph_stats

# Paths
LABELS_DIR = Path("/Users/maheshkundurthi/Documents/Research work OPT/CholecT50/labels")
PROJECT_ROOT = Path("..").resolve()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CSV_DIR = OUTPUT_DIR / "parsed_triplets"
GRAPH_DIR = OUTPUT_DIR / "graphs"
STATS_DIR = OUTPUT_DIR / "stats"
SEQ_DIR = OUTPUT_DIR / "temporal_sequences"

for d in [CSV_DIR, GRAPH_DIR, STATS_DIR, SEQ_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Labels dir: {LABELS_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
json_files = sorted(LABELS_DIR.glob("VID*.json"))
print(f"Found {len(json_files)} video JSONs")

Labels dir: /Users/maheshkundurthi/Documents/Research work OPT/CholecT50/labels
Output dir: /Users/maheshkundurthi/Documents/Research work OPT/causal-knowledge-graph-surgical-ai/outputs
Found 50 video JSONs


## 1. Parse All 50 CholecT50 Videos

Run every video JSON through `triplet_parser.parse_video()`, filter to valid triplets, and save per-video CSVs plus a combined CSV.

In [14]:
all_dfs = []
parse_results = []

for json_path in json_files:
    vid_name = json_path.stem
    t0 = time.time()
    
    df_raw = parse_video(json_path)
    df_valid = filter_valid_triplets(df_raw)
    elapsed = time.time() - t0
    
    # Save per-video CSV
    csv_path = CSV_DIR / f"{vid_name}_triplets.csv"
    df_valid.to_csv(csv_path, index=False)
    
    all_dfs.append(df_valid)
    parse_results.append({
        "video": vid_name,
        "raw_rows": len(df_raw),
        "valid_rows": len(df_valid),
        "frames": df_valid["frame"].nunique(),
        "unique_triplets": df_valid["triplet_id"].nunique(),
        "time_sec": round(elapsed, 2),
    })

df_all = pd.concat(all_dfs, ignore_index=True)
df_all.to_csv(CSV_DIR / "all_triplets.csv", index=False)

df_parse = pd.DataFrame(parse_results)
print(f"Parsed {len(df_parse)} videos, {len(df_all)} total valid rows")
df_parse

Parsed 50 videos, 150927 total valid rows


,video,raw_rows,valid_rows,frames,unique_triplets,time_sec
0,VID01,2751,2560,1543,22,0.09
1,VID02,4293,4073,2620,34,0.01
2,VID04,2331,2037,1229,20,0.01
3,VID05,3459,2911,1797,26,0.01
4,VID06,3813,3747,2088,30,0.01
5,VID08,2217,1949,1252,26,0.01
6,VID10,3674,3479,1555,26,0.01
7,VID103,3587,3031,1664,22,0.01
8,VID110,3053,2524,1648,27,0.01
9,VID111,3298,2769,1617,29,0.01


## 2. Build Graphs for All Videos

Construct a NetworkX `MultiDiGraph` for every video and save as GEXF.

In [15]:
all_graph_stats = []

for vid_name, df_vid in df_all.groupby("video"):
    G = build_graph(df_vid, video_name=vid_name)
    save_graph(G, GRAPH_DIR / f"{vid_name}_graph.gexf")
    stats = graph_stats(G)
    all_graph_stats.append(stats)

df_gstats = pd.DataFrame(all_graph_stats)
df_gstats.to_csv(STATS_DIR / "graph_stats_all.csv", index=False)
print(f"Built {len(df_gstats)} graphs")
df_gstats[["video", "nodes", "n_instruments", "n_targets", "edges", "density", "total_frames"]]

  Saved VID01_graph.gexf
  Saved VID02_graph.gexf
  Saved VID04_graph.gexf
  Saved VID05_graph.gexf
  Saved VID06_graph.gexf
  Saved VID08_graph.gexf
  Saved VID10_graph.gexf
  Saved VID103_graph.gexf
  Saved VID110_graph.gexf
  Saved VID111_graph.gexf
  Saved VID12_graph.gexf
  Saved VID13_graph.gexf
  Saved VID14_graph.gexf
  Saved VID15_graph.gexf
  Saved VID18_graph.gexf
  Saved VID22_graph.gexf
  Saved VID23_graph.gexf
  Saved VID25_graph.gexf
  Saved VID26_graph.gexf
  Saved VID27_graph.gexf
  Saved VID29_graph.gexf
  Saved VID31_graph.gexf
  Saved VID32_graph.gexf
  Saved VID35_graph.gexf
  Saved VID36_graph.gexf
  Saved VID40_graph.gexf
  Saved VID42_graph.gexf
  Saved VID43_graph.gexf
  Saved VID47_graph.gexf
  Saved VID48_graph.gexf
  Saved VID49_graph.gexf
  Saved VID50_graph.gexf
  Saved VID51_graph.gexf
  Saved VID52_graph.gexf
  Saved VID56_graph.gexf
  Saved VID57_graph.gexf
  Saved VID60_graph.gexf
  Saved VID62_graph.gexf
  Saved VID65_graph.gexf
  Saved VID66_graph.ge

,video,nodes,n_instruments,n_targets,edges,density,total_frames
0,VID01,15,6,9,19,0.35,1420
1,VID02,17,6,11,28,0.42,2522
2,VID04,13,5,8,17,0.42,1174
3,VID05,18,6,12,23,0.32,1736
4,VID06,16,6,10,26,0.43,2058
5,VID08,15,6,9,22,0.41,1191
6,VID10,15,6,9,23,0.43,1431
7,VID103,13,5,8,17,0.42,1637
8,VID110,16,6,10,21,0.35,1501
9,VID111,16,6,10,25,0.42,1502


## 3. Dataset-Wide Statistics

### 3.1 Global Summary

In [16]:
summary = video_summary(df_all)
summary.to_csv(STATS_DIR / "video_summary_all.csv", index=False)

ml_stats = multi_label_analysis(df_all)

global_stats = {
    "total_videos": df_all["video"].nunique(),
    "total_valid_rows": len(df_all),
    "unique_triplet_classes": df_all["triplet_id"].nunique(),
    "unique_instruments": df_all["instrument"].nunique(),
    "unique_verbs": df_all["verb"].nunique(),
    "unique_targets": df_all["target"].nunique(),
    "instruments": sorted(df_all["instrument"].unique().tolist()),
    "verbs": sorted(df_all["verb"].unique().tolist()),
    "targets": sorted(df_all["target"].unique().tolist()),
    "multi_label_pct": ml_stats["multi_label_pct"],
    "max_triplets_per_frame": ml_stats["max_triplets_per_frame"],
    "mean_triplets_per_frame": ml_stats["mean_triplets_per_frame"],
}

with open(STATS_DIR / "global_dataset_stats.json", "w") as f:
    json.dump(global_stats, f, indent=2)

for k, v in global_stats.items():
    if not isinstance(v, list):
        print(f"  {k}: {v}")

  total_videos: 50
  total_valid_rows: 150927
  unique_triplet_classes: 100
  unique_instruments: 6
  unique_verbs: 10
  unique_targets: 15
  multi_label_pct: 60.5
  max_triplets_per_frame: 3
  mean_triplets_per_frame: 1.68


### 3.2 Per-Video Frame Distribution

In [17]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frames per video
summary_sorted = summary.sort_values("total_frames", ascending=True)
axes[0].barh(summary_sorted["video"], summary_sorted["total_frames"], color="#2196F3", alpha=0.8)
axes[0].set_xlabel("Frames")
axes[0].set_title("Frames per Video")
axes[0].tick_params(axis="y", labelsize=7)

# Triplet rows per video
axes[1].barh(summary_sorted["video"], summary_sorted["total_triplet_rows"], color="#4CAF50", alpha=0.8)
axes[1].set_xlabel("Triplet Rows")
axes[1].set_title("Valid Triplet Rows per Video")
axes[1].tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.savefig(STATS_DIR / "frames_per_video.png", dpi=120, bbox_inches="tight")
plt.show()

/var/folders/f9/lh9gvd9902jc59075zkmypvc0000gn/T/ipykernel_70685/1566737202.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.3 Triplet Class Distribution

In [18]:
triplet_dist = df_all.groupby("triplet_label").size().sort_values(ascending=False)
triplet_dist.to_csv(STATS_DIR / "triplet_class_distribution.csv", header=["count"])

fig, ax = plt.subplots(figsize=(14, 6))
top_30 = triplet_dist.head(30)
ax.barh(range(len(top_30)), top_30.values, color="#FF7043", alpha=0.85)
ax.set_yticks(range(len(top_30)))
ax.set_yticklabels(top_30.index, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Frequency (total rows)")
ax.set_title("Top 30 Triplet Classes by Frequency")
plt.tight_layout()
plt.savefig(STATS_DIR / "triplet_class_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Total triplet classes: {len(triplet_dist)}")
print(f"Top 5:\n{top_30.head()}")
print(f"\nBottom 5:\n{triplet_dist.tail()}")

Total triplet classes: 100
Top 5:
triplet_label
grasper,retract,gallbladder    41565
hook,dissect,gallbladder       29477
grasper,retract,liver          12975
hook,dissect,cystic_duct        7891
grasper,grasp,gallbladder       7171
dtype: int64

Bottom 5:
triplet_label
irrigator,dissect,cystic_plate    10
bipolar,retract,cystic_pedicle     9
hook,coagulate,cystic_plate        9
bipolar,retract,cystic_duct        8
bipolar,grasp,cystic_plate         8
dtype: int64


/var/folders/f9/lh9gvd9902jc59075zkmypvc0000gn/T/ipykernel_70685/1716259474.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.4 Phase Distribution

In [19]:
if "phase" in df_all.columns:
    phase_dist = df_all.groupby("phase").agg(
        videos=("video", "nunique"),
        frames=("frame", "nunique"),
        rows=("triplet_id", "count"),
    ).sort_values("rows", ascending=False)
    phase_dist.to_csv(STATS_DIR / "phase_distribution.csv")
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(phase_dist.index, phase_dist["rows"], color="#AB47BC", alpha=0.8)
    ax.set_xlabel("Total Triplet Rows")
    ax.set_title("Triplet Rows per Surgical Phase")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(STATS_DIR / "phase_distribution.png", dpi=120, bbox_inches="tight")
    plt.show()
    
    phase_dist

/var/folders/f9/lh9gvd9902jc59075zkmypvc0000gn/T/ipykernel_70685/3404823858.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 3.5 Instrument and Verb Usage

In [20]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Instrument usage
inst_dist = df_all.groupby("instrument").size().sort_values(ascending=True)
axes[0].barh(inst_dist.index, inst_dist.values, color="#2196F3", alpha=0.8)
axes[0].set_title("Instrument Usage (total rows)")

# Verb usage
verb_dist = df_all.groupby("verb").size().sort_values(ascending=True)
axes[1].barh(verb_dist.index, verb_dist.values, color="#FF9800", alpha=0.8)
axes[1].set_title("Verb Usage (total rows)")

plt.tight_layout()
plt.savefig(STATS_DIR / "instrument_verb_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

/var/folders/f9/lh9gvd9902jc59075zkmypvc0000gn/T/ipykernel_70685/2810375861.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Temporal Graph Sequences

For each video, build a chronological sequence of graph states. Each frame gets:
- A **multi-hot vector** (100-d) over all triplet classes
- An **edge list** with instrument/verb/target indices
- The **surgical phase** label

This is the training data format for the models in Week 2+.

In [21]:
# Build label mappings
all_triplet_labels = sorted(df_all["triplet_label"].unique().tolist())
triplet_to_idx = {label: idx for idx, label in enumerate(all_triplet_labels)}
num_classes = len(all_triplet_labels)

instrument_labels = sorted(df_all["instrument"].unique().tolist())
verb_labels = sorted(df_all["verb"].unique().tolist())
target_labels = sorted(df_all["target"].unique().tolist())

entity_mapping = {
    "instruments": {label: idx for idx, label in enumerate(instrument_labels)},
    "verbs": {label: idx for idx, label in enumerate(verb_labels)},
    "targets": {label: idx for idx, label in enumerate(target_labels)},
}

with open(SEQ_DIR / "triplet_label_mapping.json", "w") as f:
    json.dump(triplet_to_idx, f, indent=2)
with open(SEQ_DIR / "entity_label_mapping.json", "w") as f:
    json.dump(entity_mapping, f, indent=2)

print(f"Triplet classes: {num_classes}")
print(f"Instruments: {len(instrument_labels)} -> {instrument_labels}")
print(f"Verbs: {len(verb_labels)} -> {verb_labels}")
print(f"Targets: {len(target_labels)} -> {target_labels}")

Triplet classes: 100
Instruments: 6 -> ['bipolar', 'clipper', 'grasper', 'hook', 'irrigator', 'scissors']
Verbs: 10 -> ['aspirate', 'clip', 'coagulate', 'cut', 'dissect', 'grasp', 'irrigate', 'null_verb', 'pack', 'retract']
Targets: 15 -> ['abdominal_wall_cavity', 'adhesion', 'blood_vessel', 'cystic_artery', 'cystic_duct', 'cystic_pedicle', 'cystic_plate', 'fluid', 'gallbladder', 'gut', 'liver', 'null_target', 'omentum', 'peritoneum', 'specimen_bag']


In [22]:
for vid_name, df_vid in df_all.groupby("video"):
    frames = sorted(df_vid["frame"].unique())
    sequence_data = []
    
    for frame_id in frames:
        frame_df = df_vid[df_vid["frame"] == frame_id]
        
        # Multi-hot encoding
        multi_hot = np.zeros(num_classes, dtype=np.int8)
        active_indices = []
        for label in frame_df["triplet_label"].unique():
            if label in triplet_to_idx:
                idx = triplet_to_idx[label]
                multi_hot[idx] = 1
                active_indices.append(idx)
        
        # Edge-level detail
        edges = []
        for _, row in frame_df.iterrows():
            edges.append({
                "instrument": row["instrument"],
                "instrument_idx": entity_mapping["instruments"].get(row["instrument"], -1),
                "verb": row["verb"],
                "verb_idx": entity_mapping["verbs"].get(row["verb"], -1),
                "target": row["target"],
                "target_idx": entity_mapping["targets"].get(row["target"], -1),
                "triplet_label": row["triplet_label"],
                "triplet_idx": triplet_to_idx.get(row["triplet_label"], -1),
            })
        
        phase = frame_df["phase"].iloc[0] if "phase" in frame_df.columns else "unknown"
        
        sequence_data.append({
            "frame": int(frame_id),
            "phase": phase,
            "num_active_triplets": len(active_indices),
            "active_triplet_indices": active_indices,
            "multi_hot": multi_hot.tolist(),
            "edges": edges,
        })
    
    with open(SEQ_DIR / f"{vid_name}_sequence.json", "w") as f:
        json.dump(sequence_data, f)
    
    print(f"  {vid_name}: {len(sequence_data)} frames")

print(f"\nAll sequences saved to {SEQ_DIR}")

  VID01: 1543 frames
  VID02: 2620 frames
  VID04: 1229 frames
  VID05: 1797 frames
  VID06: 2088 frames
  VID08: 1252 frames
  VID10: 1555 frames
  VID103: 1664 frames
  VID110: 1648 frames
  VID111: 1617 frames
  VID12: 863 frames
  VID13: 929 frames
  VID14: 1485 frames
  VID15: 1778 frames
  VID18: 1823 frames
  VID22: 1434 frames
  VID23: 1541 frames
  VID25: 2028 frames
  VID26: 1566 frames
  VID27: 2021 frames
  VID29: 2113 frames
  VID31: 3803 frames
  VID32: 1909 frames
  VID35: 2044 frames
  VID36: 2157 frames
  VID40: 1913 frames
  VID42: 3488 frames
  VID43: 2011 frames
  VID47: 2096 frames
  VID48: 1739 frames
  VID49: 1432 frames
  VID50: 1031 frames
  VID51: 2704 frames
  VID52: 1869 frames
  VID56: 1574 frames
  VID57: 2245 frames
  VID60: 2129 frames
  VID62: 1960 frames
  VID65: 1616 frames
  VID66: 1704 frames
  VID68: 1714 frames
  VID70: 1144 frames
  VID73: 1208 frames
  VID74: 1573 frames
  VID75: 1732 frames
  VID78: 647 frames
  VID79: 3244 frames
  VID80: 1636

## 5. Spot-Check: Sample Temporal Sequence

Pick a random video and inspect its first and last frame to verify correctness.

In [23]:
sample_vid = "VID14"
with open(SEQ_DIR / f"{sample_vid}_sequence.json") as f:
    seq = json.load(f)

print(f"{sample_vid}: {len(seq)} frames\n")
print("First frame:")
print(json.dumps(seq[0], indent=2)[:600])
print(f"\nLast frame:")
print(json.dumps(seq[-1], indent=2)[:600])

# Verify multi-hot dimensions
assert len(seq[0]["multi_hot"]) == num_classes, "Multi-hot dimension mismatch!"
print(f"\nMulti-hot dimension: {len(seq[0]['multi_hot'])} (expected {num_classes}) OK")

VID14: 1485 frames

First frame:
{
  "frame": 11,
  "phase": "preparation",
  "num_active_triplets": 1,
  "active_triplet_indices": [
    49
  ],
  "multi_hot": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
 

Last frame:
{
  "frame": 1695,
  "phase": "cleaning-and-coagulation",
  "num_active_triplets": 1,
  "active_triplet_indices": [
    51
  ],
  "multi_hot": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0

## 6. Train / Val / Test Splits

70% train / 15% val / 15% test, seeded for reproducibility.

In [24]:
all_videos = sorted(df_all["video"].unique().tolist())
np.random.seed(42)
shuffled = np.random.permutation(all_videos).tolist()

n = len(shuffled)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

splits = {
    "train": sorted(shuffled[:n_train]),
    "val": sorted(shuffled[n_train:n_train + n_val]),
    "test": sorted(shuffled[n_train + n_val:]),
}

with open(OUTPUT_DIR / "data_splits.json", "w") as f:
    json.dump(splits, f, indent=2)

print(f"Train: {len(splits['train'])} videos")
print(f"Val:   {len(splits['val'])} videos")
print(f"Test:  {len(splits['test'])} videos")

# Show split details
for split_name, vids in splits.items():
    split_df = df_all[df_all["video"].isin(vids)]
    print(f"\n{split_name.upper()}: {len(vids)} videos, "
          f"{split_df['frame'].nunique()} unique frame IDs, "
          f"{len(split_df)} rows")
    print(f"  Videos: {vids}")

Train: 35 videos
Val:   7 videos
Test:  8 videos

TRAIN: 35 videos, 3913 unique frame IDs, 109622 rows
  Videos: ['VID01', 'VID02', 'VID05', 'VID06', 'VID08', 'VID10', 'VID110', 'VID111', 'VID13', 'VID14', 'VID15', 'VID22', 'VID23', 'VID25', 'VID27', 'VID31', 'VID36', 'VID40', 'VID42', 'VID43', 'VID48', 'VID49', 'VID50', 'VID51', 'VID52', 'VID56', 'VID60', 'VID62', 'VID66', 'VID70', 'VID75', 'VID78', 'VID79', 'VID80', 'VID92']

VAL: 7 videos, 2440 unique frame IDs, 19495 rows
  Videos: ['VID04', 'VID12', 'VID32', 'VID35', 'VID57', 'VID68', 'VID74']

TEST: 8 videos, 2302 unique frame IDs, 21810 rows
  Videos: ['VID103', 'VID18', 'VID26', 'VID29', 'VID47', 'VID65', 'VID73', 'VID96']


## 7. Week 1 Summary

| Output | Location |
|--------|----------|
| Per-video CSVs | `outputs/parsed_triplets/VIDxx_triplets.csv` |
| Combined CSV | `outputs/parsed_triplets/all_triplets.csv` |
| Graph files (GEXF) | `outputs/graphs/VIDxx_graph.gexf` |
| Statistics | `outputs/stats/` |
| Temporal sequences | `outputs/temporal_sequences/VIDxx_sequence.json` |
| Label mappings | `outputs/temporal_sequences/*_mapping.json` |
| Data splits | `outputs/data_splits.json` |

**Next: Week 2** - Build LSTM and Transformer baselines for next-action prediction.